# Bacteria analysis

## Import data

In [1]:
!qiime tools import \
  --type 'SampleData[PairedEndSequencesWithQuality]' \
  --input-path data \
  --input-format CasavaOneEightSingleLanePerSampleDirFmt \
  --output-path raw_input.qza


!qiime demux summarize \
     --i-data raw_input.qza \
     --p-n 10000 \
     --verbose \
     --o-visualization raw_input.qzv

Imported data as CasavaOneEightSingleLanePerSampleDirFmt to raw_input.qza
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  context['result_data'] = pd.concat([context['result_data'], df])
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['301 nts' '301 nts' '301 nts' '301 nts' '301 nts' '301 nts' '301 nts']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  stats[stats.index != 'count'] = \
/Users/ac04

## Remove primers, only keep reads with primers

In [2]:
!qiime cutadapt trim-paired \
  --i-demultiplexed-sequences raw_input.qza \
  --p-cores 30 \
  --p-front-f NTGYCAGCMGCCGCGGTAA \
  --p-front-r NCGYCAATTYMTTTRAGTTT \
  --p-discard-untrimmed \
  --o-trimmed-sequences trimmed_seqs.qza \
  --verbose > trim.log

Done           00:00:00        57,717 reads @  12.6 µs/read;   4.75 M reads/minute
Done           00:00:00        46,108 reads @  13.9 µs/read;   4.30 M reads/minute
Done           00:00:00        52,608 reads @  12.4 µs/read;   4.84 M reads/minute
Done           00:00:00        39,921 reads @  15.8 µs/read;   3.79 M reads/minute
Done           00:00:00        47,160 reads @  14.0 µs/read;   4.28 M reads/minute
Done           00:00:00        56,871 reads @  12.5 µs/read;   4.79 M reads/minute
Done           00:00:00        42,053 reads @  15.8 µs/read;   3.79 M reads/minute
Done           00:00:00        51,951 reads @  13.0 µs/read;   4.62 M reads/minute
Done           00:00:00        60,204 reads @  11.6 µs/read;   5.17 M reads/minute
Done           00:00:00        51,115 reads @  13.3 µs/read;   4.51 M reads/minute
Done           00:00:00        34,140 reads @  19.1 µs/read;   3.15 M reads/minute
Done           00:00:00        34,218 reads @  19.9 µs/read;   3.01 M reads/minute
Done

In [3]:
!qiime demux summarize \
     --i-data trimmed_seqs.qza \
     --p-n 10000 \
     --verbose \
     --o-visualization trimmed_seqs.qzv

/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  context['result_data'] = pd.concat([context['result_data'], df])
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['282 nts' '282 nts' '282 nts' '282 nts' '282 nts' '282 nts' '282 nts']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  stats[stats.index != 'count'] = \
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_s

## Denoise/deduplicate merge seqs

In [4]:
!qiime dada2 denoise-paired \
    --i-demultiplexed-seqs trimmed_seqs.qza \
    --p-trunc-len-f  280 \
    --p-trunc-len-r 210 \
    --p-n-threads 10 \
    --o-table 2020_dada_table.qza \
    --o-representative-sequences 2020_dada_sequences.qza \
    --o-denoising-stats 2020_dada_stats.qza \
    --o-base-transition-stats 2020_dada_transitions.qza \
    --verbose >> dada.log



Loading required package: Rcpp
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  series[missing.index] = missing
/Users/ac044747/anaconda

In [6]:
!qiime metadata tabulate \
    --m-input-file 2020_dada_stats.qza \
    --o-visualization 2020_dada_stats.qzv

!qiime feature-table summarize \
    --i-table 2020_dada_table.qza \
    --output-dir dadaRes \
    --m-metadata-file bacteriaMap_2020.txt

Saved Visualization to: 2020_dada_stats.qzv
Saved ImmutableMetadata to: dadaRes/feature_frequencies.qza
Saved ImmutableMetadata to: dadaRes/sample_frequencies.qza
Saved Visualization to: dadaRes/summary.qzv


## Filter

Remove the Clermontia samples as these were not present in the 2023 data and other samples.

In [7]:
!qiime feature-table filter-samples \
  --i-table 2020_dada_table.qza \
  --m-metadata-file bacteriaMap_2020.txt \
  --p-where "Plant!='Clermontia sp.' AND Preferred!='NA'" \
  --o-filtered-table 2020_dada_table_filter_samples.qza

!qiime feature-table summarize \
    --i-table 2020_dada_table_filter_samples.qza \
    --output-dir dadaRes2 \
    --m-metadata-file bacteriaMap_2020.txt

Saved FeatureTable[Frequency] to: 2020_dada_table_filter_samples.qza
Saved ImmutableMetadata to: dadaRes2/feature_frequencies.qza
Saved ImmutableMetadata to: dadaRes2/sample_frequencies.qza
Saved Visualization to: dadaRes2/summary.qzv


Remove ASVs that are present in a single sample or that have a total frequency lower than 0.1% of the average read depth of the samples (~20, average reads per sample = 15,938)

In [8]:
!qiime feature-table filter-features \
  --i-table 2020_dada_table_filter_samples.qza \
  --p-min-samples 2 \
  --p-min-frequency 20 \
  --o-filtered-table 2020_filtered_table.qza
!qiime feature-table summarize \
    --i-table 2020_filtered_table.qza \
    --output-dir dadaFiltered \
    --m-metadata-file bacteriaMap_2020.txt

Saved FeatureTable[Frequency] to: 2020_filtered_table.qza
Saved ImmutableMetadata to: dadaFiltered/feature_frequencies.qza
Saved ImmutableMetadata to: dadaFiltered/sample_frequencies.qza
Saved Visualization to: dadaFiltered/summary.qzv


In [9]:
!qiime feature-table filter-seqs \
  --i-data 2020_dada_sequences.qza \
  --i-table 2020_filtered_table.qza \
  --o-filtered-data 2020_filtered_sequences.qza


Saved FeatureData[Sequence] to: 2020_filtered_sequences.qza
